In [3]:
"""

This module computes comprehensive evaluation metrics for outlier detection algorithms.
It calculates standard metrics (Accuracy, Precision, Recall) and advanced metrics 
(AUC, Average Precision, Precision@n) adjusted for outlier imbalance.

Organization:
  1. Imports & Dependencies
  2. Dataset Configuration
  3. Basic Metrics Functions (Accuracy, Precision, Recall, F1)
  4. Advanced Metrics Functions (AUC, AP, P@n)
  5. Adjusted Metrics (normalized by class imbalance)
  6. Aggregation & Computation Functions
  7. Main Execution Pipeline
"""

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np
import os
import re
import warnings
from tqdm import tqdm

In [ ]:
# Suppress library warnings for cleaner output
warnings.filterwarnings("ignore")


# =============================================================================
# SECTION 1: Dataset Configuration
# =============================================================================

# Define dataset files organized by category (with sample counts as comments)
PATH_DATASETS = r'..\..\datasets\processed'
FILES_DATASET = {
    'literature': os.listdir(os.sep.join([PATH_DATASETS, 'literature'])),
    'semantic': os.listdir(os.sep.join([PATH_DATASETS, 'semantic'])),
    'odds': os.listdir(os.sep.join([PATH_DATASETS, 'odds'])),
    'ADBench': os.listdir(os.sep.join([PATH_DATASETS, 'ADBench'])),
    'real': os.listdir(os.sep.join([PATH_DATASETS, 'real'])),
}

# =============================================================================
# SECTION 2: Data Preparation Helper
# =============================================================================

def prepare_input(lista_output: pd.Series, lista_type: pd.Series, 
                  scope: str = None) -> tuple:
    """
    Convert output (correctness) and type ('I'/'O') columns to binary labels.
    
    This function transforms boolean correctness values and type strings into
    binary classification targets suitable for sklearn metrics.
    
    Args:
        lista_output: Series of boolean correctness values (True/False)
        lista_type: Series of type strings ('I' for inlier, 'O' for outlier)
        scope: Filter scope:
               - None or 'I': y_true=1 for inliers
               - 'O': y_true=1 for outliers
               
    Returns:
        Tuple of (y_true, y_pred) as binary lists [0, 1]
    """
    # Convert correctness to binary predictions (1=correct, 0=incorrect)
    y_pred = list(map(lambda x: 1 if x else 0, lista_output.tolist()))
    
    # Convert type strings to binary labels based on scope
    if scope is None or scope == 'I':
        # Inlier scope: 1 if type is 'I', else 0
        y_true = list(map(lambda x: 1 if x == 'I' else 0, lista_type.tolist()))
    elif scope == 'O':
        # Outlier scope: 1 if type is 'O', else 0
        y_true = list(map(lambda x: 1 if x == 'O' else 0, lista_type.tolist()))
    
    return y_true, y_pred

In [5]:
# =============================================================================
# SECTION 3: Basic Classification Metrics
# =============================================================================

def compute_accuracy(y_true: list, y_pred: list) -> float:
    """
    Calculate accuracy (fraction of correct predictions).
    
    Accuracy = (TP + TN) / (TP + TN + FP + FN)
    
    Args:
        y_true: Ground truth binary labels
        y_pred: Predicted binary labels
        
    Returns:
        Accuracy score in [0, 1]
    """
    return accuracy_score(y_true, y_pred)


def compute_precision(y_true: list, y_pred: list) -> float:
    """
    Calculate precision (fraction of positive predictions that are correct).
    
    Precision = TP / (TP + FP)
    
    Args:
        y_true: Ground truth binary labels
        y_pred: Predicted binary labels
        
    Returns:
        Precision score in [0, 1]
    """
    return precision_score(y_true, y_pred, zero_division=0)


def compute_recall(y_true: list, y_pred: list) -> float:
    """
    Calculate recall (fraction of positive labels correctly identified).
    
    Recall = TP / (TP + FN)
    
    Args:
        y_true: Ground truth binary labels
        y_pred: Predicted binary labels
        
    Returns:
        Recall score in [0, 1]
    """
    return recall_score(y_true, y_pred, zero_division=0)


def compute_f1(y_true: list, y_pred: list) -> float:
    """
    Calculate F1-score (harmonic mean of precision and recall).
    
    F1 = 2 * (Precision * Recall) / (Precision + Recall)
    
    Args:
        y_true: Ground truth binary labels
        y_pred: Predicted binary labels
        
    Returns:
        F1-score in [0, 1]
    """
    return f1_score(y_true, y_pred, zero_division=0)


# =============================================================================
# SECTION 4: Advanced Ranking-Based Metrics
# =============================================================================

def compute_auc_score(actual: list, predicted: list) -> float:
    """
    Calculate Area Under the ROC Curve (AUROC/AUC).
    
    AUROC measures the model's ability to distinguish between positive and negative
    classes across all possible classification thresholds.
    
    Args:
        actual: Ground truth binary labels (0 or 1)
        predicted: Predicted anomaly scores (higher = more anomalous)
        
    Returns:
        AUC score in [0, 1]; 0.5 = random classifier, 1.0 = perfect classifier
    """
    try:
        return roc_auc_score(actual, predicted)
    except ValueError:
        # Return 0.5 if only one class present (undefined AUC)
        return 0.5


def compute_average_precision(actual: list, predicted: list, 
                              sorted_indices: list) -> float:
    """
    Calculate Average Precision (AP) from precision-recall curve.
    
    AP integrates precision values across recall points where positives are ranked.
    Higher scores indicate better ranking of positive instances.
    
    Args:
        actual: Ground truth binary labels (0 or 1)
        predicted: Predicted anomaly scores (higher = more anomalous)
        sorted_indices: Pre-computed indices sorted by prediction scores (descending)
        
    Returns:
        Average Precision in [0, 1]
    """
    tp = 0  # True positives (outliers correctly ranked high)
    fp = 0  # False positives (inliers incorrectly ranked high)
    precision_values = []
    recall_values = []
    
    # Iterate through ranked instances from highest to lowest score
    for i in range(len(sorted_indices)):
        if actual[sorted_indices[i]] == 1:
            tp += 1
        else:
            fp += 1
        
        # Compute precision and recall at each threshold
        precision = tp / (tp + fp)
        recall = tp / sum(actual) if sum(actual) > 0 else 0
        precision_values.append(precision)
        recall_values.append(recall)
    
    # Calculate area under precision-recall curve using trapezoid rule
    ap = 0.0
    for i in range(1, len(sorted_indices)):
        # Area = (recall_change) * (precision at higher recall)
        ap += (recall_values[i] - recall_values[i - 1]) * precision_values[i]
    
    return ap


def compute_precision_at_k(actual: list, predicted: list, n: int) -> float:
    """
    Calculate Precision@n: fraction of top-n predictions that are correct.
    
    P@n = (True Positives in top-n) / n
    
    For outlier detection, typically n = number of outliers in dataset (R-Precision).
    
    Args:
        actual: Ground truth binary labels (0 or 1)
        predicted: Predicted anomaly scores (higher = more anomalous)
        n: Number of top-ranked instances to evaluate
        
    Returns:
        Precision@n in [0, 1]
    """
    if n <= 0 or n > len(predicted):
        return 0.0
    
    # Sort indices by prediction scores (descending)
    sorted_indices = sorted(range(len(predicted)), 
                           key=lambda i: predicted[i], 
                           reverse=True)
    
    tp = 0  # Count of true outliers in top-n
    
    # Count true positives in top-n predictions
    for i in range(min(n, len(sorted_indices))):
        if actual[sorted_indices[i]] == 1:
            tp += 1
    
    return tp / n if n > 0 else 0.0


def compute_max_f1_score(y_true: list, y_pred: list) -> float:
    """
    Find the maximum F1-score by exploring different classification thresholds.
    
    Useful for continuous anomaly scores where the optimal threshold is unknown.
    Tries thresholds from 0.1 to 0.9 and returns the best F1-score achieved.
    
    Args:
        y_true: Ground truth binary labels (0 or 1)
        y_pred: Predicted anomaly scores in [0, 1]
        
    Returns:
        Maximum F1-score across all thresholds
    """
    thresholds = np.arange(0.1, 1.0, 0.1)
    f1_scores = []
    
    # Test each threshold
    for threshold in thresholds:
        y_pred_binary = [1 if score > threshold else 0 for score in y_pred]
        f1 = compute_f1(y_true, y_pred_binary)
        f1_scores.append(f1)
    
    # Return best F1 (or 0 if no valid scores)
    return np.max(f1_scores) if f1_scores else 0.0


# =============================================================================
# SECTION 5: Adjusted Metrics (Normalized by Class Imbalance)
# =============================================================================

def adjust_average_precision(ap: float, num_outliers: int, 
                             num_instances: int) -> float:
    """
    Adjust Average Precision by random baseline for imbalanced datasets.
    
    Normalization removes the effect of class imbalance:
    Adjusted AP = (AP - baseline) / (1 - baseline)
    
    where baseline = outlier_fraction (random classifier performance).
    
    Args:
        ap: Original Average Precision score
        num_outliers: Number of outlier instances in dataset
        num_instances: Total number of instances
        
    Returns:
        Adjusted AP in [0, 1] (roughly; can be negative if AP < baseline)
    """
    baseline = num_outliers / num_instances if num_instances > 0 else 0
    denominator = 1 - baseline
    
    if denominator <= 0:
        return 0.0
    
    return (ap - baseline) / denominator


def adjust_r_precision(r_precision: float, num_outliers: int, 
                       num_instances: int) -> float:
    """
    Adjust Precision@n (R-Precision) by random baseline for imbalanced datasets.
    
    Accounts for the fact that random guessing would achieve
    (num_outliers / num_instances) accuracy in imbalanced settings.
    
    Adjusted P@n = (P@n - baseline) / (1 - baseline)
    
    Args:
        r_precision: Original Precision@n (P@R where R=num_outliers)
        num_outliers: Number of outlier instances in dataset
        num_instances: Total number of instances
        
    Returns:
        Adjusted P@n in [0, 1] (roughly; can be negative if P@n < baseline)
    """
    baseline = num_outliers / num_instances if num_instances > 0 else 0
    denominator = 1 - baseline
    
    if denominator <= 0:
        return 0.0
    
    return (r_precision - baseline) / denominator


def adjust_maximum_f1(max_f1: float, num_outliers: int, 
                      num_instances: int) -> float:
    """
    Adjust Maximum F1-score by random baseline for imbalanced datasets.
    
    Removes the effect of class imbalance on F1 performance.
    
    Adjusted MaxF1 = (MaxF1 - baseline) / (1 - baseline)
    
    Args:
        max_f1: Original Maximum F1-score
        num_outliers: Number of outlier instances in dataset
        num_instances: Total number of instances
        
    Returns:
        Adjusted Max F1 in [0, 1] (roughly; can be negative if MaxF1 < baseline)
    """
    baseline = num_outliers / num_instances if num_instances > 0 else 0
    denominator = 1 - baseline
    
    if denominator <= 0:
        return 0.0
    
    return (max_f1 - baseline) / denominator

In [ ]:
# =============================================================================
# SECTION 6: Aggregation & Aggregation Functions
# =============================================================================

def calc_metrics(results_df: pd.DataFrame, dataset: str, 
                scope: str = None) -> tuple:
    """
    Calculate average metrics (Accuracy, Precision, Recall) for a dataset.
    
    Aggregates metrics across all algorithms and parameters tested on a dataset.
    For each algorithm-parameter combination, computes the requested metric,
    then averages across algorithms.
    
    Args:
        results_df: Detailed execution results DataFrame with columns
                    [algorithm, parameter, correct, type, ...]
        dataset: Name of dataset to filter
        scope: Metric scope ('O' for outliers, 'I' for inliers, None for both)
        
    Returns:
        Tuple of (dataset_name, avg_accuracy, avg_precision, avg_recall)
    """
    result_accuracy = []
    result_precision = []
    result_recall = []
    
    # Filter to current dataset
    df_dataset = results_df.query('dataset == @dataset')
    if len(df_dataset) == 0:
        return dataset, 0, 0, 0
    
    # Iterate over algorithms
    for algorithm in df_dataset['algorithm'].unique():
        df_algorithm = df_dataset.query('algorithm == @algorithm')
        
        tmp_acc = []
        tmp_prec = []
        tmp_rec = []
        
        # Iterate over parameter values
        for parameter in df_algorithm['parameter'].unique():
            # Build query with optional scope filter
            query_str = f'parameter == @parameter'
            if scope is not None:
                query_str += f' and type == @scope'
            
            df_parameter = df_algorithm.query(query_str)
            
            # Prepare binary labels and predictions
            y_true, y_pred = prepare_input(df_parameter['correct'], 
                                          df_parameter['type'], 
                                          scope)
            
            # Compute metrics for this parameter
            if len(y_true) > 0:
                tmp_acc.append(compute_accuracy(y_true, y_pred))
                tmp_prec.append(compute_precision(y_true, y_pred))
                tmp_rec.append(compute_recall(y_true, y_pred))
        
        # Filter NaN values and aggregate by algorithm
        tmp_acc = [x for x in tmp_acc if not np.isnan(x)]
        tmp_prec = [x for x in tmp_prec if not np.isnan(x)]
        tmp_rec = [x for x in tmp_rec if not np.isnan(x)]
        
        if tmp_acc:
            result_accuracy.append(np.mean(tmp_acc))
        if tmp_prec:
            result_precision.append(np.mean(tmp_prec))
        if tmp_rec:
            result_recall.append(np.mean(tmp_rec))
    
    # Average across all algorithms
    avg_acc = np.mean(result_accuracy) if result_accuracy else 0
    avg_prec = np.mean(result_precision) if result_precision else 0
    avg_rec = np.mean(result_recall) if result_recall else 0
    
    return dataset, round(avg_acc, 4), round(avg_prec, 4), round(avg_rec, 4)


def calc_metrics_adjusted(results_df: pd.DataFrame, dataset: str, group: str, 
                         scope: str = None) -> tuple:
    """
    Calculate adjusted ranking-based metrics for a dataset.
    
    Computes AUC, Precision@n (R-Precision), Average Precision and Max-F1, all adjusted
    for class imbalance. This gives more realistic performance estimates for
    imbalanced outlier detection tasks.
    
    Args:
        results_df: Detailed execution results DataFrame
        dataset: Name of dataset to filter
        group: Dataset category (used in output)
        scope: Unused legacy parameter (kept for compatibility)
        
    Returns:
        Tuple of (clean_dataset_name, group, avg_auc, avg_p_at_n, avg_ap, avg_max_f1)
    """
    result_auc = []
    result_p_at_n = []
    result_avg_prec = []
    result_max_f1 = []
    
    # Filter to current dataset
    df_dataset = results_df.query('dataset == @dataset')
    if len(df_dataset) == 0:
        return dataset, group, 0, 0, 0
    
    # Iterate over algorithms
    for algorithm in df_dataset['algorithm'].unique():
        df_algorithm = df_dataset.query('algorithm == @algorithm')
        
        tmp_auc = []
        tmp_p_at_n = []
        tmp_avg_prec = []
        tmp_max_f1 = []
        
        # Iterate over parameter values
        for parameter in df_algorithm['parameter'].unique():
            df_parameter = df_algorithm.query('parameter == @parameter')
            
            # Convert type column to binary (1 if outlier, 0 otherwise)
            Y = list(map(lambda x: 1 if x == 'O' else 0, 
                        df_parameter['type'].tolist()))
            
            if len(Y) == 0 or sum(Y) == 0:
                continue
            
            # Extract anomaly scores and compute counts
            scores = df_parameter['score'].tolist()
            n_outliers = sum(Y)
            n_instances = len(df_parameter)
            
            # Compute AUC
            auc = compute_auc_score(Y, scores)
            tmp_auc.append(auc)
            
            # Compute Precision@n (R-Precision)
            r_precision = compute_precision_at_k(Y, scores, n_outliers)
            adjusted_r_prec = adjust_r_precision(r_precision, 
                                                 n_outliers, 
                                                 n_instances)
            tmp_p_at_n.append(adjusted_r_prec)
            
            # Compute Average Precision
            sorted_indices = sorted(range(len(scores)), 
                                   key=lambda i: scores[i], 
                                   reverse=True)
            ap = compute_average_precision(Y, scores, sorted_indices)
            adjusted_ap = adjust_average_precision(ap, n_outliers, n_instances)
            tmp_avg_prec.append(adjusted_ap)
            
            # Compute Max-F1
            max_f1 = compute_max_f1_score(Y, scores)
            adjusted_max_f1= adjust_maximum_f1(max_f1, n_outliers, n_instances)
            tmp_max_f1.append(adjusted_max_f1)
        
        # Filter NaN and aggregate by algorithm
        tmp_auc = [x for x in tmp_auc if not np.isnan(x)]
        tmp_p_at_n = [x for x in tmp_p_at_n if not np.isnan(x)]
        tmp_avg_prec = [x for x in tmp_avg_prec if not np.isnan(x)]
        tmp_max_f1 = [x for x in tmp_max_f1 if not np.isnan(x)]
        
        if tmp_auc:
            result_auc.append(np.mean(tmp_auc))
        if tmp_p_at_n:
            result_p_at_n.append(np.mean(tmp_p_at_n))
        if tmp_avg_prec:
            result_avg_prec.append(np.mean(tmp_avg_prec))
        if tmp_max_f1:
            result_max_f1.append(np.mean(tmp_max_f1))
    
    # Average across all algorithms
    avg_auc = np.mean(result_auc) if result_auc else 0
    avg_p_at_n = np.mean(result_p_at_n) if result_p_at_n else 0
    avg_ap = np.mean(result_avg_prec) if result_avg_prec else 0
    avg_max_f1 = np.mean(result_max_f1) if result_max_f1 else 0
    
    # Clean dataset name by removing version suffixes
    clean_dataset = re.sub('_v[0-9]+', '', dataset)
    
    return clean_dataset, group, round(avg_auc, 4), round(avg_p_at_n, 4), round(avg_ap, 4), round(avg_max_f1, 4)


# =============================================================================
# SECTION 7: Main Execution Pipeline
# =============================================================================

def compute_basic_metrics(results_dir: str = r'..\..\results', 
                         output_file: str = r'..\..\metrics_outlier.csv') -> None:
    """
    Compute and save basic metrics (Accuracy, Precision, Recall) for all datasets.
    
    Processes each dataset group, calculates metrics focused on outlier detection,
    and saves results to CSV.
    
    Args:
        results_dir: Directory containing {group}_detail_execution.csv files
        output_file: Output CSV filename (saved in results_dir)
    """
    print("\n" + "="*70)
    print("Computing Basic Metrics (Accuracy, Precision, Recall)")
    print("="*70)
    
    metrics_outlier = []
    
    for group, datasets in FILES_DATASET.items():
        print(f'\nGroup: {group}')
        
        # Load detailed results for this group
        detail_file = os.path.join(results_dir, f'{group}_detail_execution.csv')
        if not os.path.exists(detail_file):
            print(f"  ⚠ File not found: {detail_file}")
            continue
        
        df = pd.read_csv(detail_file, sep=';')
        
        # Process each dataset with progress bar
        with tqdm(total=len(datasets), desc="Processing datasets", 
                 unit="it", leave=True, colour="green") as pbar:
            for dataset in datasets:
                metrics_outlier.append(calc_metrics(df, dataset, 'O'))
                pbar.update(1)
    
    # Save results
    df_outlier = pd.DataFrame(metrics_outlier, 
                             columns=['dataset', 'accuracy', 'precision', 'recall'])
    output_path = os.path.join(results_dir, output_file)
    df_outlier.to_csv(output_path, sep=';', index=False)
    print(f"\n✓ Basic metrics saved to: {output_path}")


def compute_adjusted_metrics(results_dir: str = r'..\..\results', 
                            output_file: str = r'..\..\metrics.csv') -> pd.DataFrame:
    """
    Compute and save adjusted ranking-based metrics for all datasets.
    
    Calculates AUC, Precision@n, Average Precision and Max-F1 (all adjusted for
    class imbalance), aggregates by dataset and method, and saves results.
    
    Args:
        results_dir: Directory containing {group}_detail_execution.csv files
        output_file: Output CSV filename (saved in results_dir)
        
    Returns:
        DataFrame of aggregated metrics
    """
    print("\n" + "="*70)
    print("Computing Adjusted Ranking-Based Metrics (AUC, P@n, AP, Max-F1)")
    print("="*70)
    
    metrics_geral = []
    
    for group, datasets in FILES_DATASET.items():
        print(f'\nGroup: {group}')
        
        # Load detailed results for this group
        detail_file = os.path.join(results_dir, f'{group}_detail_execution.csv')
        if not os.path.exists(detail_file):
            print(f"File not found: {detail_file}")
            continue
        
        df = pd.read_csv(detail_file, sep=';')
        
        # Process each dataset with progress bar
        with tqdm(total=len(datasets), desc="Processing datasets", 
                 unit="it", leave=True, colour="green") as pbar:
            for dataset in datasets:
                metrics_geral.append(calc_metrics_adjusted(df, dataset, group))
                pbar.update(1)
    
    # Create DataFrame
    df_metrics = pd.DataFrame(metrics_geral, 
                             columns=['dataset', 'method', 'AUC', 'P@n', 'AP', 'Max-F1'])
    
    # Aggregate by dataset and method (in case of duplicates)
    df_metrics = df_metrics.groupby(['dataset', 'method'], as_index=False).agg({
        'AUC': 'mean',
        'P@n': 'mean',
        'AP': 'mean',
        'Max-F1': 'mean',
    }).sort_values(by=['method', 'dataset']).reset_index(drop=True)
    
    # Save results
    output_path = os.path.join(results_dir, output_file)
    df_metrics.to_csv(output_path, sep=';', index=False)
    print(f"\n✓ Adjusted metrics saved to: {output_path}")
    
    return df_metrics

In [9]:
def main():
    """
    Main execution: compute all metrics and save results.
    
    Executes two pipelines:
    1. Basic metrics: Accuracy, Precision, Recall (outlier-focused)
    2. Adjusted metrics: AUC, P@n, AP, Max-F1 (normalized for class imbalance)
    """
    results_directory = r'..\results'
    
    # Ensure results directory exists
    os.makedirs(results_directory, exist_ok=True)
    
    # Compute basic metrics for outlier detection
    compute_basic_metrics(results_dir=results_directory, 
                         output_file='metrics_outlier.csv')
    
    # Compute adjusted ranking-based metrics
    df_metrics_final = compute_adjusted_metrics(results_dir=results_directory, 
                                               output_file='metrics.csv')
    
    print("\n" + "="*70)
    print("Metrics Computation Complete")
    print("="*70)
    print("\nMetrics Summary (first 10 rows):")
    print(df_metrics_final.head(10))


if __name__ == "__main__":
    main()


Computing Basic Metrics (Accuracy, Precision, Recall)

Group: literature


Processing datasets: 100%|██████████| 10/10 [00:31<00:00,  3.12s/it]



Group: semantic
  ⚠ File not found: ..\results\semantic_detail_execution.csv

Group: odds
  ⚠ File not found: ..\results\odds_detail_execution.csv

Group: ADBench
  ⚠ File not found: ..\results\ADBench_detail_execution.csv

Group: real
  ⚠ File not found: ..\results\real_detail_execution.csv

✓ Basic metrics saved to: ..\results\metrics_outlier.csv

Computing Adjusted Ranking-Based Metrics (AUC, P@n, AP, Max-F1)

Group: literature


Processing datasets: 100%|██████████| 10/10 [01:36<00:00,  9.63s/it]


Group: semantic
File not found: ..\results\semantic_detail_execution.csv

Group: odds
File not found: ..\results\odds_detail_execution.csv

Group: ADBench
File not found: ..\results\ADBench_detail_execution.csv

Group: real
File not found: ..\results\real_detail_execution.csv

✓ Adjusted metrics saved to: ..\results\metrics.csv

Metrics Computation Complete

Metrics Summary (first 10 rows):
          dataset      method      AUC      P@n       AP   Max-F1
0  Ionosphere.csv  literature  0.81496  0.44079  0.40961  0.20951
